# Lesson 4 — Structured Output with Pydantic

## Goal

Understanding:

- Why valid JSON is not enough for production AI systems.
- What Pydantic is  and why LangChain uses it.
- What  `BaseModel` represents.
- The difference between dictionaries and Pydantic models.
- How schemas improve reliability.
- How `PydanticOutputParser` generates structured output from LLMs.
- When to use `JsonOutputParser` vs `PydanticOutputParser`.

In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load the environment
load_dotenv()

True

In [3]:
# Read the model name
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [4]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [ ]:
# creating pydantic model
from pydantic import BaseModel, Field

class Country(BaseModel): # By inheriting from `BaseModel` we are telling pydantic: this class describes the structure of some data
    name: str
    capital: str

country = Country( # country is an object (an instance) of the Country class
    name = "Egypt",
    capital = "Cairo"
)

In [6]:
print(type(country))

<class '__main__.Country'>


In [ ]:
# In pydantic we access fields like  a normal object:

print(country.name) # not country["name"]
print(country.capital)

Egypt
Cairo


In [8]:
# Convert from object to dict or JSON 

country_dict  = country.model_dump() # model_dump() convert from object into dict
country_JSON  = country.model_dump_json() # model_dump_json() convert from object into JSON
print(country_dict)
print(country_JSON)

{'name': 'Egypt', 'capital': 'Cairo'}
{"name":"Egypt","capital":"Cairo"}


In [9]:
# Convert from dict into object

data = {
    "name": "Egypt",
    "capital": "Cairo"
}

country = Country (**data) # `**` tells python to: "Take every key-value pair in this dictionary and path it as keyword arguments."
country


Country(name='Egypt', capital='Cairo')

## The complete picture of dict and object

```text
Python dict
      │
      │ Country(**data)
      ▼
Country object
      │
      │ model_dump()
      ▼
Python dict
      │
      │ model_dump_json()
      ▼
JSON text
```

##  Dict vs JSON

- Python dict: an object in python memory.
- JSON: a text representation used to exchange data between systems.

In [12]:
# Mistake we made intentionally.  

"""country = Country(
    name=123,
    capital="Cairo"
)"""

'country = Country(\n    name=123,\n    capital="Cairo"\n)'

Pydantic is better than the dict in LangChain as it validates the attributes and raises a validation error if the validation is failed 

In [13]:
from langchain_core.output_parsers  import PydanticOutputParser

In [15]:
# Define the schema

class Country(BaseModel):
    """Information about a country"""

    name: str = Field(
        description="The common name of the country"
    )

    capital: str = Field(
        description="The  capital city of the country"
    )

    population: int = Field(
        description="Approximate population"
    )

In [16]:
# Create the parser

parser = PydanticOutputParser(
    pydantic_object= Country  # Means this line should produce `country` objects
)


In [17]:
print (parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Information about a country", "properties": {"name": {"description": "The common name of the country", "title": "Name", "type": "string"}, "capital": {"description": "The  capital city of the country", "title": "Capital", "type": "string"}, "population": {"description": "Approximate population", "title": "Population", "type": "integer"}}, "required": ["name", "capital", "population"]}
```


In [ ]:
# Create the template

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template = """
Answer the following question.

{format_instructions}

Question:
{question}
""",
    input_variables= ["question"],
    partial_variables = {
        "format_instructions": parser.get_format_instructions()
    }
)

In [19]:
print(prompt.invoke({"question":"Tell me about Egypt"}).text)


Answer the following question.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Information about a country", "properties": {"name": {"description": "The common name of the country", "title": "Name", "type": "string"}, "capital": {"description": "The  capital city of the country", "title": "Capital", "type": "string"}, "population": {"description": "Approximate population", "title": "Population", "type": "integer"}}, "required": ["name", "capital", "population"]}
```

Question:
Tell me about Egypt



In [21]:
# Build the prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a helpful assistant.
            {format_instructions}
            """,
        ),
        ("human", "{question}")
    ]
). partial(format_instructions = parser.get_format_instructions()) 

In [22]:
# Render the prompt

prompt_value = prompt.invoke(
    {
        "question": "Tell me about Egypt"
    }
)
prompt_value

ChatPromptValue(messages=[SystemMessage(content='\n            You are a helpful assistant.\n            The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "Information about a country", "properties": {"name": {"description": "The common name of the country", "title": "Name", "type": "string"}, "capital": {"description": "The  capital city of the country", "title": "Capital", "type": "string"}, "population": {"description": "Approximate population", "title": "Population", "type": "integer"}}, "required": ["name", "capital", "population"]}\n```\n            ', additional_

In [23]:
# Invoke the model
response = llm.invoke(prompt_value)

In [24]:
print(type(response)) # without the parser
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
[{'type': 'text', 'text': '{"name": "Egypt", "capital": "Cairo", "population": 112000000}', 'extras': {'signature': 'EoEJCv4IARFNMg/Bb6p8g5s0V17pgxq/rsI/kalUG5wN2uYs2W579AL8qa61gCgMRq3pxoJMrfSuTk0JzQ4A9e8W7dUx3CUxhJscHary2oSk8dPA5nkilmm90O/Xw0RGWzPezQEHdSlv0Rz1wPB+i2UtZAXZ6Rii34J8D6Sia9KF09OyiD40BVDQV3FoCHlD1CyjNdOZUo/g2o3CrTsYvn1IpqIP75o0u48h26+CW4vS/6A8WmbGrzf8Bo7IOD6VELkqnNO7cSSPTeB4+a+wpTARq0/lZyVwzOV3KIJEEnKRYs1IEH6ug4urAEeYVAilDGQ+oB6UXgYTwAYBgQrumo5uxlbLScYEDoo4P7CEt6bTJ4KKn3hcUsloatljx53eKn3IN1jsJjQeaFJmyOLHdqCPMQynJDpfY7s0cLKOIPvyoHob2ZX0JJRDPUoweHWl2+lNT+JBJfszJNbRS4NCGus9QilHgMERXTMQ7RzA5+CEZkyF0Wa8BI6Ntc+1LSDB5e4U6vqUNW8Spz7BLGZ2xYo6rAvYpohK8Z6wAmJlE3TLAYZH/L1nMsWLrA6ncm7tbCU5ksP5Q9yI1KfkCjoMe0MBekQNTk8fqCfVE0r6NwVlgsdL4QjqI0n8PCnqYB5rxm0Rjq53GXjK11TYMKjrXs6AogjVgBhdtwnAM0dR/tUDA7ddilYMrEf0MA3jVdjuF7Y7tf4ET4hi0iakEPl/RomjPecudTDrDMNCzXc43sNlx+e/2OFtwuksupisRL/iTeNCYOBMltabVQ1udGoYUtTHnEApL1zQc/QsNOkO8qWiHoSdv8MLKf8inkFZNwJ/qLPp

In [25]:
result = parser.invoke (response)

print(type(result))
print(result)

<class '__main__.Country'>
name='Egypt' capital='Cairo' population=112000000


## Nested Pydantic Models

In [27]:
class President(BaseModel):
    name: str = Field(
        description="The full name of the president"
    )
    age: int = Field(
        description="The age of the president"
    )

In [28]:
# Define the schema

class Country(BaseModel):
    """Information about a country"""

    name: str = Field(
        description="The common name of the country"
    )

    capital: str = Field(
        description="The capital city of the country"
    )

    population: int = Field(
        description="Approximate population"
    )

    president: President

In [29]:
# Create the parser

parser = PydanticOutputParser(
    pydantic_object= Country  # Means this line should produce `country` objects
)

In [30]:
print (parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"President": {"properties": {"name": {"description": "The full name of the president", "title": "Name", "type": "string"}, "age": {"description": "The age of the president", "title": "Age", "type": "integer"}}, "required": ["name", "age"], "title": "President", "type": "object"}}, "description": "Information about a country", "properties": {"name": {"description": "The common name of the country", "title": "Name", "type": "string"}, "capital": {"description": "The capital city of the country", "title": "Capital", "type": "string"}, 

In [31]:
# Create the template

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template = """
Answer the following question.

{format_instructions}

Question:
{question}
""",
    input_variables= ["question"],
    partial_variables = {
        "format_instructions": parser.get_format_instructions()
    }
)

In [32]:
# Build the prompt

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a helpful assistant.
            {format_instructions}
            """,
        ),
        ("human", "{question}")
    ]
). partial(format_instructions = parser.get_format_instructions()) 

In [33]:
# Render the prompt

prompt_value = prompt.invoke(
    {
        "question": "Tell me about Egypt"
    }
)
prompt_value

ChatPromptValue(messages=[SystemMessage(content='\n            You are a helpful assistant.\n            The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"$defs": {"President": {"properties": {"name": {"description": "The full name of the president", "title": "Name", "type": "string"}, "age": {"description": "The age of the president", "title": "Age", "type": "integer"}}, "required": ["name", "age"], "title": "President", "type": "object"}}, "description": "Information about a country", "properties": {"name": {"description": "The common name of the country", "title": "Name", "type": 

In [34]:
# Invoke the model
response = llm.invoke(prompt_value)

In [35]:
print(type(response)) # without the parser
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
[{'type': 'text', 'text': '```json\n{\n  "name": "Egypt",\n  "capital": "Cairo",\n  "population": 112000000,\n  "president": {\n    "name": "Abdel Fattah el-Sisi",\n    "age": 69\n  }\n}\n```', 'extras': {'signature': 'EpUOCpIOARFNMg+77PQsdYunIsEvhV/uC/2p46s4SM7/kVKMi5N59EkD9EiH2jMUmuJjGOq9X0t4uFbjhQGxFGiI8Z18YxrfaTm4RurHS2MHNEDiLXXDNN9powrrNOXx222JkQBPDSqQRK4tpNeLKv1k9joKI651yxuWsxdBr6/muynt1dBJymTsYKCX8YVGcRP5i5ysR7feKt3LVP0Lt7+EQNw3fNsSlJiD0Eaa89+1Gkp0cNk2ZZcJkVBaQ1c8PiuJCRvyQ5RoYNeGA2qZVl+1WlMsWd17eKJD1wqxBV51OnVS1kc0HBUa0FXr5MCdAy2EZ86EuLTgrcBE33B3rQPyB9L/9lVApp2IpqJdODu1eG/UUrV9X7maP1L8KXwkSC4ROaGzIs2Ht2HQL1R2rI3lMzoXazmYVcE3ta4rBrii3gY2q0LWXQlKR7IrQ/848aGR8LiRtSp+c9c2BQcxAndZXdTKbktI2mwSiUCjeFIrZG/7TIDnExyCHx2XSFSCIlkeVpE2HxNNthaC8oVBBNwrFsc2eUEvKWSvJw/0H3ztcNzO9lgCu0MUvvRmWKCfkmpG0Znss/FmDi6T+rf03zSda5tg/Zg9IfNV1mgx+oUQN/Wpxq73Y3P4BnFi6CQLcIld8ZJHWOx4Y3LHFWK7SKl/jqB/lvpCBjhgTxZEUvQjcFuovwe7eXbFTdrjwMJx5hoixmcgc30kd/v0IFOgVWh85dqjkm

In [36]:
result = parser.invoke (response)

print(type(result))
print(result)

<class '__main__.Country'>
name='Egypt' capital='Cairo' population=112000000 president=President(name='Abdel Fattah el-Sisi', age=69)


In [37]:
# Convert result into dict
dict_result = result.model_dump()
dict_result

{'name': 'Egypt',
 'capital': 'Cairo',
 'population': 112000000,
 'president': {'name': 'Abdel Fattah el-Sisi', 'age': 69}}